# 02 — Analysis

**Purpose:** turn raw snapshots into the analysis-ready dataset and compute every headline result.

**Inputs:** `data/markets_with_snapshots.csv` (from notebook 01).

**Outputs:** `output/polymarket_breaks_dataset.csv`, plus result tables in `output/`.

**Prerequisites:** run `01-data-collection` first.

The dataset is rebuilt here from cached raw by `build_dataset()`. Delete the committed CSV and re-run: you get the same file back.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

In [ ]:
import csv
from src.transform import build_dataset
from src import analysis as A
from src.utils import bracket_pattern, infer_category, SNAPSHOTS

### Rebuild the analysis-ready dataset from raw
This is the transform step: selection rule, derived columns, mechanical labels. It writes `output/polymarket_breaks_dataset.csv`.

In [ ]:
dataset = build_dataset(write=True)
print(f'{len(dataset)} breaks in the dataset')

### Sample composition
Full sample, core sample (Iran cluster removed), and the cluster itself.

In [ ]:
summary = A.sample_summary(dataset)
for k, v in summary.items():
    print(f'{k:>22}: {v:,.0f}' if 'usd' in k else f'{k:>22}: {v}')

### Direction of breaks
Dismissed outcomes landing (NO that resolved YES) versus confident favorites collapsing (YES that resolved NO).

In [ ]:
A.direction_split(dataset)

### Break RATE and the denominator
The rate is breaks divided by every market that reached high confidence, so it needs the snapshot rows, not just the breaks. The function returns both counts.

In [ ]:
snaps = list(csv.DictReader(open(SNAPSHOTS, encoding='utf-8')))
A.break_rate(snaps)

### Relative risk: bracket markets versus everything else
The structural finding. Bracket markets on a numerical outcome break at a higher rate than all other high-confidence markets.

In [ ]:
def is_bracket(r):
    cat = (r.get('category') or '').strip() or infer_category(
        f"{r.get('event_title','')} {r.get('market_question','')}")
    return bracket_pattern(cat, r.get('event_title',''),
                           r.get('market_question','')) != 'other'
A.relative_risk(snaps, is_bracket)

### Concentration
Bracket-pattern markets as a share of break count and of exposure.

In [ ]:
A.bracket_concentration(dataset)

### Breakdowns by category and structural pattern

In [ ]:
by_cat = A.by_category(dataset)
for r in by_cat:
    print(f"{r['category']:<14}{r['breaks']:>5}  ${r['exposure_usd']/1e6:>6.1f}M")

In [ ]:
A.by_bracket_pattern(dataset)

### Robustness: confidence threshold and the boundary rule

In [ ]:
A.threshold_sensitivity(snaps)

In [ ]:
A.boundary_robustness(snaps)

### Save result tables
Small summary tables are committed to `output/`; raw data is not.

In [ ]:
import csv as _csv
from src.utils import OUTPUT_DIR
OUTPUT_DIR.mkdir(exist_ok=True)
with open(OUTPUT_DIR / 'by-category.csv', 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=by_cat[0].keys()); w.writeheader(); w.writerows(by_cat)
print('wrote output/by-category.csv')